# 02 — K-Means Independiente por Régimen  (Secciones 5-7)
## Predicción de Deserción Estudiantil · Equipo 7

---
**Prerrequisito:** `01_preprocessing_feature_engineering.ipynb`

Cubre las secciones 5 a 7 del pipeline original:
- **§5** Segmentación por régimen (PreTec21 / Tec21)
- **§6** K-Means **independiente** por régimen + selección K óptimo (codo, Silhouette, DB)
- **§7** Match de clusters entre regímenes por distancia coseno

Guarda en `data/processed/`: `df_pre.csv`, `df_tec.csv`, `df_match.csv` (DataFrames),
más modelos KMeans y scalers en pickle.


## Setup e Importaciones

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import pickle
import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.spatial.distance import cosine as cosine_dist

from sklearn.cluster       import KMeans
from sklearn.metrics       import silhouette_score, davies_bouldin_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

SEED = 42
np.random.seed(SEED)
print('✓ Librerías cargadas')

## Configuración

In [ ]:
PROCESSED_DIR = Path('../../data/processed')
IMG_DIR       = Path('images'); IMG_DIR.mkdir(exist_ok=True)

PRETEC21_GENS  = ['AD14', 'AD15', 'AD16', 'AD17', 'AD18']
TEC21_GENS     = ['AD19', 'AD20']
TARGET         = 'retention'
K_CLUSTERS     = 4
MIN_SILHOUETTE = 0.30
MAX_DB         = 1.50
MATCH_HIGH     = 0.15
MATCH_PARTIAL  = 0.35

print(f'K_CLUSTERS: {K_CLUSTERS}')
print('NOTA: K-Means se entrena INDEPENDIENTEMENTE en cada régimen.')

## Carga del Dataset Procesado

In [ ]:
df = pd.read_csv(PROCESSED_DIR / 'df_preprocessed.csv')
print(f'✓ df cargado  →  {df.shape}')
print(f'Generaciones: {sorted(df["generation"].unique())}')

## §5 — Segmentación por Régimen

In [ ]:
df_pre = df[df['generation'].isin(PRETEC21_GENS)].copy().reset_index(drop=True)
df_tec = df[df['generation'].isin(TEC21_GENS)].copy().reset_index(drop=True)

print(f'PreTec21 (AD14-AD18): {len(df_pre):,} estudiantes  |  Deserción: {(df_pre[TARGET]==0).mean()*100:.1f}%')
print(f'Tec21    (AD19-AD20): {len(df_tec):,} estudiantes  |  Deserción: {(df_tec[TARGET]==0).mean()*100:.1f}%')

## §6 — K-Means INDEPENDIENTE por Régimen

**Diferencia fundamental:** se ajusta un K-Means **separado** para cada régimen.
Cada modelo encuentra los clusters óptimos dentro de su propia distribución,
sin heredar la geometría del otro régimen.

In [ ]:
CLUSTER_COLS_CANDIDATES = [
    'PNA', 'admission_test_norm', 'english.evaluation', 'admission.rubric',
    'apoyo_financiero', 'has_extracurriculars', 'first_gen_enc',
    'educ_padres_max', 'FTE', 'age',
]
CLUSTER_COLS = [c for c in CLUSTER_COLS_CANDIDATES if c in df.columns]
print(f'Variables de clustering ({len(CLUSTER_COLS)}): {CLUSTER_COLS}')

## §6b — Selección de K Óptimo (Codo + Silhouette + Davies-Bouldin)

Se evalúa k ∈ {2,…,8} en ambos regímenes de forma independiente usando una
submuestra de 5 000 puntos para velocidad. El k se elige donde Silhouette es máximo.

In [ ]:
K_RANGE     = range(2, 9)
SAMPLE_SIZE = 5000

def select_k(X_scaled, regime_name):
    idx = np.random.choice(len(X_scaled), min(SAMPLE_SIZE, len(X_scaled)), replace=False)
    X_s = X_scaled[idx]
    inertias, sils, dbs = [], [], []
    for k in K_RANGE:
        km_tmp = KMeans(n_clusters=k, n_init=10, random_state=SEED)
        lab    = km_tmp.fit_predict(X_s)
        inertias.append(km_tmp.inertia_)
        sils.append(silhouette_score(X_s, lab))
        dbs.append(davies_bouldin_score(X_s, lab))
        print(f'  {regime_name}  k={k}  inertia={km_tmp.inertia_:>8,.0f}'
              f'  sil={sils[-1]:.3f}  DB={dbs[-1]:.3f}')
    best_sil_k = list(K_RANGE)[int(np.argmax(sils))]
    best_db_k  = list(K_RANGE)[int(np.argmin(dbs))]
    print(f'  → k óptimo por Silhouette: {best_sil_k}  |  por DB: {best_db_k}')
    return inertias, sils, dbs

X_pre_df_tmp = df_pre[CLUSTER_COLS].copy().fillna(df_pre[CLUSTER_COLS].median())
X_tec_df_tmp = df_tec[CLUSTER_COLS].copy().fillna(df_tec[CLUSTER_COLS].median())
Xp_tmp = StandardScaler().fit_transform(X_pre_df_tmp.values.astype(float))
Xt_tmp = StandardScaler().fit_transform(X_tec_df_tmp.values.astype(float))

print('=== PreTec21 ===')
iner_pre, sil_pre_ks, db_pre_ks = select_k(Xp_tmp, 'PreTec21')
print()
print('=== Tec21 ===')
iner_tec, sil_tec_ks, db_tec_ks = select_k(Xt_tmp, 'Tec21')

# Visualización
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
ks = list(K_RANGE)
for row, (name, inertias, sils, dbs) in enumerate([
    ('PreTec21', iner_pre, sil_pre_ks, db_pre_ks),
    ('Tec21',    iner_tec, sil_tec_ks, db_tec_ks)
]):
    axes[row,0].plot(ks, inertias, 'o-', color='#2563eb')
    axes[row,0].axvline(K_CLUSTERS, ls='--', color='red', alpha=0.5, label=f'k={K_CLUSTERS}')
    axes[row,0].set_title(f'Método del Codo — {name}')
    axes[row,0].set_xlabel('k'); axes[row,0].set_ylabel('Inercia')
    axes[row,0].legend(); axes[row,0].grid(alpha=0.3)

    axes[row,1].plot(ks, sils, 'o-', color='#16a34a')
    axes[row,1].axhline(MIN_SILHOUETTE, ls='--', color='gray', label=f'Min={MIN_SILHOUETTE}')
    axes[row,1].axvline(K_CLUSTERS, ls='--', color='red', alpha=0.5, label=f'k={K_CLUSTERS}')
    axes[row,1].set_title(f'Silhouette Score — {name}')
    axes[row,1].set_xlabel('k'); axes[row,1].legend(); axes[row,1].grid(alpha=0.3)

    axes[row,2].plot(ks, dbs, 'o-', color='#dc2626')
    axes[row,2].axhline(MAX_DB, ls='--', color='gray', label=f'Max={MAX_DB}')
    axes[row,2].axvline(K_CLUSTERS, ls='--', color='red', alpha=0.5, label=f'k={K_CLUSTERS}')
    axes[row,2].set_title(f'Davies-Bouldin — {name}')
    axes[row,2].set_xlabel('k'); axes[row,2].legend(); axes[row,2].grid(alpha=0.3)

plt.suptitle('Selección de K Óptimo por Régimen (K-Means Independiente)', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(IMG_DIR / 'k_selection_independiente.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\n✓ Guardado: {IMG_DIR}/k_selection_independiente.png')
print(f'K configurado: {K_CLUSTERS}')

### K-Means PreTec21

In [ ]:
X_pre_df = df_pre[CLUSTER_COLS].copy()
X_pre_df.fillna(X_pre_df.median(), inplace=True)

scaler_pre = StandardScaler()
Xp_scaled  = scaler_pre.fit_transform(X_pre_df.values.astype(float))

km_pre = KMeans(n_clusters=K_CLUSTERS, n_init=20, random_state=SEED)
km_pre.fit(Xp_scaled)
df_pre['cluster'] = km_pre.labels_

sil_pre = silhouette_score(Xp_scaled, km_pre.labels_)
db_pre  = davies_bouldin_score(Xp_scaled, km_pre.labels_)
print(f'PreTec21 → Silhouette={sil_pre:.3f}  DB={db_pre:.3f}')
print(f'{"✓" if sil_pre>=MIN_SILHOUETTE else "✗"} Silhouette  |  {"✓" if db_pre<=MAX_DB else "✗"} DB')

### K-Means Tec21 (scaler propio — independiente)

In [ ]:
X_tec_df = df_tec[CLUSTER_COLS].copy()
X_tec_df.fillna(X_tec_df.median(), inplace=True)

scaler_tec = StandardScaler()
Xt_scaled  = scaler_tec.fit_transform(X_tec_df.values.astype(float))

km_tec = KMeans(n_clusters=K_CLUSTERS, n_init=20, random_state=SEED)
km_tec.fit(Xt_scaled)
df_tec['cluster'] = km_tec.labels_

sil_tec = silhouette_score(Xt_scaled, km_tec.labels_)
db_tec  = davies_bouldin_score(Xt_scaled, km_tec.labels_)
print(f'Tec21 → Silhouette={sil_tec:.3f}  DB={db_tec:.3f}')
print(f'{"✓" if sil_tec>=MIN_SILHOUETTE else "✗"} Silhouette  |  {"✓" if db_tec<=MAX_DB else "✗"} DB')

### Perfiles de clusters

In [ ]:
def cluster_profile(df_regime, regime_name):
    stats = (df_regime.groupby('cluster')
             .agg(n=(TARGET,'count'),
                  tasa_desercion=(TARGET, lambda x: (x==0).mean()))
             .round(3))
    stats['pct_total'] = (stats['n'] / len(df_regime) * 100).round(1)
    print(f'\n═══ Perfiles {regime_name} (K-Means independiente) ═══')
    print(stats.sort_values('tasa_desercion', ascending=False).to_string())
    return stats

stats_pre = cluster_profile(df_pre, 'PreTec21')
stats_tec = cluster_profile(df_tec, 'Tec21')

In [ ]:
print('═══ Medias por cluster (PreTec21) ═══')
for c in sorted(df_pre['cluster'].unique()):
    sub = df_pre[df_pre['cluster']==c]
    print(f'\nCluster {c} | n={len(sub):,} | dropout={(sub[TARGET]==0).mean():.3f}')
    for col in CLUSTER_COLS:
        print(f'  {col:<30} {sub[col].mean():.4f}')

In [ ]:
print('═══ Medias por cluster (Tec21) ═══')
for c in sorted(df_tec['cluster'].unique()):
    sub = df_tec[df_tec['cluster']==c]
    print(f'\nCluster {c} | n={len(sub):,} | dropout={(sub[TARGET]==0).mean():.3f}')
    for col in CLUSTER_COLS:
        print(f'  {col:<30} {sub[col].mean():.4f}')

### PCA 2D — visualización de clusters

In [ ]:
pca_pre = PCA(n_components=2, random_state=SEED).fit(Xp_scaled)
pca_tec = PCA(n_components=2, random_state=SEED).fit(Xt_scaled)
Zp = pca_pre.transform(Xp_scaled)
Zt = pca_tec.transform(Xt_scaled)

COLORS = plt.cm.tab10.colors
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for name, Z, labels, pca_obj, ax in [
    ('PreTec21', Zp, km_pre.labels_, pca_pre, axes[0]),
    ('Tec21',    Zt, km_tec.labels_, pca_tec, axes[1])
]:
    for k in range(K_CLUSTERS):
        mask = labels == k
        ax.scatter(Z[mask,0], Z[mask,1], s=15, alpha=0.35, color=COLORS[k], label=f'Cluster {k}')
    ax.set_title(f'{name} (K-Means independiente)')
    ax.set_xlabel(f'PC1 ({pca_obj.explained_variance_ratio_[0]*100:.1f}%)')
    ax.set_ylabel(f'PC2 ({pca_obj.explained_variance_ratio_[1]*100:.1f}%)')
    ax.legend(markerscale=2, fontsize=8); ax.grid(alpha=0.2)

plt.suptitle('Clusters PCA 2D — K-Means Independiente', fontsize=13)
plt.tight_layout()
plt.savefig(IMG_DIR / 'clusters_pca_independiente.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'✓ Guardado: {IMG_DIR}/clusters_pca_independiente.png')

## §7 — Match de Clusters entre Regímenes

Los IDs de cluster no son directamente comparables: se calcula la **distancia coseno**
entre centroides estandarizados.
- < 0.15 → **Match alto**
- 0.15–0.35 → **Match parcial**
- > 0.35 → **Sin match**

In [ ]:
centroids_pre = km_pre.cluster_centers_
centroids_tec = km_tec.cluster_centers_

dist_matrix = np.zeros((K_CLUSTERS, K_CLUSTERS))
for i in range(K_CLUSTERS):
    for j in range(K_CLUSTERS):
        dist_matrix[i, j] = cosine_dist(centroids_pre[i], centroids_tec[j])

df_dist = pd.DataFrame(dist_matrix,
                       index=[f'Pre_C{i}' for i in range(K_CLUSTERS)],
                       columns=[f'Tec_C{j}' for j in range(K_CLUSTERS)])
print('Matriz de distancias coseno:')
print(df_dist.round(4).to_string())

print('\n═══ Resultado del Match ═══')
match_results = []
for i in range(K_CLUSTERS):
    j_best = int(np.argmin(dist_matrix[i]))
    d_best = dist_matrix[i, j_best]
    quality = 'ALTO' if d_best < MATCH_HIGH else ('PARCIAL' if d_best < MATCH_PARTIAL else 'SIN MATCH')
    n_pre  = (df_pre['cluster']==i).sum()
    n_tec  = (df_tec['cluster']==j_best).sum()
    dr_pre = (df_pre[df_pre['cluster']==i][TARGET]==0).mean()
    dr_tec = (df_tec[df_tec['cluster']==j_best][TARGET]==0).mean()
    match_results.append({
        'Pre_Cluster': i, 'Tec_Cluster': j_best,
        'Cosine_dist': round(d_best,4), 'Match': quality,
        'n_Pre': n_pre, 'n_Tec': n_tec,
        'dropout_Pre': round(dr_pre,3), 'dropout_Tec': round(dr_tec,3)
    })
    print(f'Pre_C{i} → Tec_C{j_best}  |  dist={d_best:.4f}  |  {quality}')
    print(f'  PreTec21: n={n_pre:,}  dropout={dr_pre:.3f}')
    print(f'  Tec21:    n={n_tec:,}  dropout={dr_tec:.3f}')

df_match = pd.DataFrame(match_results)

# Heatmap
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(dist_matrix, cmap='YlOrRd', vmin=0, vmax=0.5)
ax.set_xticks(range(K_CLUSTERS)); ax.set_xticklabels([f'Tec_C{j}' for j in range(K_CLUSTERS)])
ax.set_yticks(range(K_CLUSTERS)); ax.set_yticklabels([f'Pre_C{i}' for i in range(K_CLUSTERS)])
for i in range(K_CLUSTERS):
    for j in range(K_CLUSTERS):
        ax.text(j, i, f'{dist_matrix[i,j]:.3f}', ha='center', va='center', fontsize=9,
                color='white' if dist_matrix[i,j] > 0.25 else 'black')
plt.colorbar(im, ax=ax)
ax.set_title('Distancia Coseno entre Clusters\n(más oscuro = más distante)')
plt.tight_layout()
plt.savefig(IMG_DIR / 'match_distance_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'✓ Guardado: {IMG_DIR}/match_distance_matrix.png')

## Guardar artefactos

In [ ]:
import json

# ── DataFrames → CSV ─────────────────────────────────────────────────────────
df_pre.to_csv(PROCESSED_DIR / 'df_pre.csv', index=False)
print(f'✓ {PROCESSED_DIR}/df_pre.csv')
df_tec.to_csv(PROCESSED_DIR / 'df_tec.csv', index=False)
print(f'✓ {PROCESSED_DIR}/df_tec.csv')
df_match.to_csv(PROCESSED_DIR / 'df_match.csv', index=False)
print(f'✓ {PROCESSED_DIR}/df_match.csv')

# ── Lista de columnas → JSON ──────────────────────────────────────────────────
with open(PROCESSED_DIR / 'cluster_cols.json', 'w') as f:
    json.dump(CLUSTER_COLS, f)
print(f'✓ {PROCESSED_DIR}/cluster_cols.json')

# ── Modelos sklearn → pickle (no tienen equivalente CSV) ─────────────────────
for fname, obj in [('km_pre.pkl', km_pre), ('km_tec.pkl', km_tec),
                   ('scaler_pre.pkl', scaler_pre), ('scaler_tec.pkl', scaler_tec)]:
    with open(PROCESSED_DIR / fname, 'wb') as f:
        pickle.dump(obj, f)
    print(f'✓ {PROCESSED_DIR}/{fname}')

# ── Arrays numpy ──────────────────────────────────────────────────────────────
np.save(PROCESSED_DIR / 'Xp_scaled.npy', Xp_scaled)
np.save(PROCESSED_DIR / 'Xt_scaled.npy', Xt_scaled)
print(f'✓ Arrays Xp_scaled / Xt_scaled guardados')

print(f'\nSilhouette  PreTec21={sil_pre:.3f}  Tec21={sil_tec:.3f}')
print(f'DB          PreTec21={db_pre:.3f}  Tec21={db_tec:.3f}')
print('\n→ Siguiente: modelos/experimentos/ o modelos/shap_rf_clusters.ipynb')